# ml.sh lease: how long a call runs before it detaches

**Keep `SH_LEASE_MS` at 15 s.**
- **Correctness:** over 135 runs, no lease moved any model's correct rate beyond the noise of 5 repeats. gemma4:31b
  and qwen3.8-flash-next:vision were 5/5 on build and quick at every lease.
- **5 s costs turns:** it spent 2.0 task turns per detached 40 s build, against 1.5 at 15 s, and 3.2 per hang against
  2.4.
- **60 s catches nothing more:** it left a stall silent four times as long, and models noticed fewer stalls before the
  host's 120 s kill (9, 8 and 6 of 15 at 5, 15 and 60 s).

**Time.**
- **Measured:** 1.9 h of wall time and 0.6 h of GPU time (the sum of `genMs`), against the 4.5–7.5 h estimate.
- **Why it was faster:** the estimate assumed every hang ran to the 120 s kill. Models answered or cancelled sooner
  (median 55–65 s per hang run). gemma4:e4b ended most cells within seconds without reaching the host (32 of its 45
  runs), and the quick arm takes about 13 s a run.

**Findings beyond the default.**
- **Unprompted estimates undershoot.** Only qwen3.8 often gave `expectMs` before a `Detached`.
  - Every estimate any model gave a 40 s build before its first `Detached` was 20–45 s, and every one of those calls
    outlived it.
  - Because the one deadline is `expectMs ?? SH_LEASE_MS`, a model's short estimate replaces the lease. That is why
    the 40 s build still detached in 44% of runs under the 60 s lease.
  - So the lease protects only calls with no estimate. The `Detached` message, not the default, is what has to carry
    a model through an underestimate. On the evidence here it does: those runs still answered right.
- **The namespace clause did not fix gemma4:e4b.** On #1437 `859cd023`, 12 of its runs still called `ml.sh('…')` as
  a function, and 20 more never reached the host in some other way. It is the only model whose failures are about
  finding `ml.sh` and not about the lease.
- **gemma4:31b rarely estimates.** It gave no `expectMs` before a `Detached` on any build under the 5 s lease, so it
  pays the full lease cost in task turns (2.6 per build at 5 s, 1.0 at 15 s).

**Negative results and limits.**
- **The quick arm does not separate 5 s from 15 s.** A 2 s call is under both, and none detached at any lease. The
  case where 5 s would tax ordinary work, a 5–15 s call such as a test run or `git fetch`, was not measured.
- **The build arm cannot measure progress.** The host has no `peek` yet, so `task tail` reports nothing while a call
  runs, and models tailed and got nothing.
- **The first matrix was invalid.** On image `18a812eb`, the scripts' `sleep` was refused by the sandbox, as shown at
  the end.
- **Four denied gates,** all qwen3.8: `ml.sh` probes outside the free tier, which the bench's approver denied by
  design. They are not a lease effect.

**The question.** `ml.sh.run` waits up to `expectMs`, or `SH_LEASE_MS` when no estimate is given, and then THROWS
`Detached {id, elapsed_ms, lease_ms, …}`. The call carries on as a task that the model reaches with the `task` tool
(`await`, `tail`, `cancel`). If the lease is too short, every ordinary call spends turns on a detach. If it is too
long, a stalled call is the silent hang again. #1437's draft default is 15 s.

**The matrix** (`tests/e2e/bench/specs/sh-lease.bench.ts`):
- three local models: gemma4:31b, gemma4:e4b, qwen3.8-flash-next:vision;
- three leases: 5, 15 and 60 s, built into each cell through `__SH_LEASE_MS__`;
- three task scripts: **build** (40 s, a progress line every 5 s, then `BUILD OK 7f3a19`), **hang** (one line, then
  never ends: the host kills it at 120 s) and **quick** (2 s, then `CHECK OK c42e07`);
- 5 repeats each.

The task text names the command and the answer wanted. It never mentions `expectMs`, leases or tasks, so an
estimate given before the first `Detached` is the model's own idea.

## Conditions

- **Code.** Integration commit `bc0edb7e`, local and unpushed:
  - window-ml main `4e5277ae`;
  - #1437 at `859cd023`: `4706fae1` plus the JSDoc clause that `ml.sh` is a namespace called as `ml.sh.run(argv)`;
  - the e2e harness for a native host, `dd870393` (#1453);
  - the bench's own commits, including `efb2eb1a`, which gives the scripts' manifest `runs-code`.
- **wmlhost:** `ae7284e`.
- **Container.** A Linux container (`tests/e2e/docker/sh-lease.Dockerfile`) with bwrap and Landlock ABI 8, run with
  seccomp, apparmor and systempaths unconfined. Every call ran `host-sandbox`.
- **#1437 was rebased during the sweep, to `47ee6834`.** The two heads' patch-ids are unequal (`d6a14a42`, `62ff40fc`)
  but their stats are identical (44 files, +1191/−77). The 12 lines that differ thread main's steering `images` field
  through `drainInbox`'s return type, its item map and the `runInboxes` queue type. The task-result branch is
  identical, and no lease, `Detached` or `ml.sh` line differs. The coordinator and the async lane agreed that this
  sweep stands for the final head.
- **Approvals.** `autoApproveReadonly` was OFF: the routing bug, #1451.
  - Every gate was ruled by `shLeaseDecide`: the run's own `exec` was approved, and every nested ask and every other
    gated tool was DENIED.
  - The scripts ran free through stored workspace manifests (effects `read`, `runs-code`), which the host pins by
    sha256.
- **Pilot.** 3/3 passed on `bc0edb7e`. Check 3 shows `bash build.sh` ran with no ask and its `sleep` ran (no
  `Permission denied`; the `SLEPT-<nonce>` line appears only after `sleep 2` succeeds).
- **Excluded.**
  - The cells from the bench fingerprint `nogit` (an image with no `.git`, which made the cache reuse cells across
    builds).
  - The b4cff4ef pilot cells: the model-facing MlSh header and JSDoc changed after them.
  - `history/`.
  - The first matrix (image `18a812eb`) is not in the tables. It is the negative result at the end.
- **What the build arm cannot measure.** The host reports a call's output only when the call ends, because it has no
  `peek` yet. So a model never sees the build's progress lines while it waits. Re-run that arm after the host's peek
  slice.

In [1]:
import json, re
from pathlib import Path
import numpy as np
import pandas as pd
import wmlnb

HERE = wmlnb.ROOT / "notebooks" / "bench"
SWEEPS = {"sh-lease": "rerun", "sh-lease-18a812eb": "first"}
EXTRACT = HERE / "data" / "sh-lease" / "runs.json"
HOST_KILL_S = 120
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 80)

## The runs, pinned

There is one row per run, read from its `cell.json` (the outcome, tokens and answer) and its `run.json` (the steps).
Each `exec` that names `ml.sh` counts as one call:
- did it give `expectMs`;
- was it before or after the run's first `Detached`;
- did it outlive its own estimate (`you expected ~N s and it is still running`).

Every `task` op is a turn the model spent on a detached call. No run received a task's result as a `[task …]` inbox
message: every model waited with `task await` inside its turn. So these task turns are the wake turns.

In [2]:
def _sweep_meta(d):
    recs = [json.loads(l) for l in (d / "sweeps.jsonl").read_text().splitlines() if l.strip()]
    sweep = [r for r in recs if r.get("op") == "sweep"][-1]
    done = json.loads((d / "done.json").read_text()) if (d / "done.json").exists() else {}
    return {"fingerprint": sweep.get("fingerprint"), "started": sweep.get("at"), "done": done.get("at"),
            "runsIn": sweep.get("runsIn")}

def _ts(s):
    return pd.Timestamp(s).timestamp()

def read_run(cell_path, sweep, meta):
    c = json.loads(cell_path.read_text())
    m = c.get("measurement") or {}
    run = json.loads((cell_path.parent / "run.json").read_text())["session"]
    steps = run["steps"]
    calls, ops = [], {"await": 0, "await_still": 0, "tail": 0, "cancel": 0}
    first_det_at = first_call_at = None
    got_result = host_killed = outlived = False
    results = []
    for s in steps:
        tool, r = s.get("tool"), str(s.get("result", ""))
        if not tool:
            continue
        results.append(r)
        js = str((s.get("arguments") or {}).get("js", ""))
        if tool == "exec" and re.search(r"\bml\.sh\b", js):
            det = "Detached:" in r
            outlived_call = det and "you expected ~" in r
            calls.append({"expect": bool(re.search(r"expectMs\s*:", js)), "before_det": first_det_at is None,
                          "detached": det, "not_fn": "ml.sh is not a function" in r,
                          "reached": det or "host-sandbox" in r or "host-plain" in r, "outlived": outlived_call})
            if calls[-1]["reached"] and first_call_at is None:
                first_call_at = _ts(s["at"])
            if det and first_det_at is None:
                first_det_at = _ts(s["at"])
            outlived |= outlived_call
            host_killed |= bool(re.search(r'"timedOut":\s*true|timedOut true', r)) and "timeoutMs" not in js
        if tool == "task":
            op = (s.get("arguments") or {}).get("op")
            if op in ops:
                ops[op] += 1
            if op == "await" and "still running" in r:
                ops["await_still"] += 1
            if "you expected it within" in r:
                outlived = True
            got_result |= " done" in r[:60]
            host_killed |= "timeoutMs" in r and "killed" in r
    answer_msgs = [x for x in run["messages"] if x["role"] == "assistant"]
    answer_at = _ts(answer_msgs[-1]["at"]) if answer_msgs else None
    gen_ms = sum((s.get("usage") or {}).get("genMs") or 0 for s in steps)
    toks = m.get("tokens") or {}
    return {
        "sweep": SWEEPS[sweep], "sweep_dir": sweep, "fingerprint": meta["fingerprint"],
        "sweep_started": meta["started"], "sweep_done": meta["done"], "runs_in": meta["runsIn"], "task": c["taskId"],
        "model": c["combo"]["model"], "lease_s": c["combo"]["leaseMs"] // 1000, "repeat": c["repeat"],
        "correct": bool(m.get("succeeded")), "hit_cap": bool(m.get("hitCap")), "steps": m.get("steps"),
        "run_s": (m.get("runMs") or 0) / 1000, "gen_s": gen_ms / 1000,
        "prompt_tokens": toks.get("prompt"), "completion_tokens": toks.get("completion"),
        "approvals": m.get("approvals"), "denied": m.get("denied"),
        "sh_calls": len(calls), "not_fn_calls": sum(x["not_fn"] for x in calls),
        "reached_host": any(x["reached"] for x in calls), "detached": first_det_at is not None,
        "expect_before_det": any(x["expect"] and x["before_det"] for x in calls if x["reached"]),
        "expect_any": any(x["expect"] for x in calls if x["reached"]),
        "outlived_estimate": outlived, "task_ops": sum(ops[k] for k in ("await", "tail", "cancel")),
        **{f"op_{k}": v for k, v in ops.items()},
        "result_via_task": got_result, "host_killed": host_killed,
        "answer_after_call_s": (answer_at - first_call_at) if (answer_at and first_call_at) else None,
        "sleep_denied": any("sleep: Permission denied" in r for r in results),
        "answer": (m.get("finalAnswer") or m.get("answer") or "")[:200],
    }

def read_sweeps():
    rows = []
    for sweep in SWEEPS:
        d = wmlnb.find_sweep(sweep)
        meta = _sweep_meta(d)
        for cell in sorted(d.glob("*/*/r*/cell.json")):
            if "history" in cell.parts or not (cell.parent / "run.json").exists():
                continue
            rows.append(read_run(cell, sweep, meta))
    return rows

data = wmlnb.extract(EXTRACT, read_sweeps, source={"sweeps": list(SWEEPS), "sweep_root": "~/git/window-ml-bench/tests/e2e/artifacts/bench"})
all_runs = pd.DataFrame(data)
assert "nogit" not in set(all_runs.fingerprint), "a nogit cell is in the extract"
runs = all_runs[all_runs.sweep == "rerun"].copy()
first = all_runs[all_runs.sweep == "first"].copy()
pd.DataFrame(wmlnb.provenance(EXTRACT))

,extract,sha256,rows,extracted_at,commit,sweep_root,sweeps
0,notebooks/bench/data/sh-lease/runs.json,76853ab95f910778,270,2026-11-07T20:13:00+00:00,bc0edb7ee349,~/git/window-ml-bench/tests/e2e/artifacts/bench,"[sh-lease, sh-lease-18a812eb]"


In [3]:
sweeps = all_runs.groupby("sweep_dir").agg(fingerprint=("fingerprint", "first"), started=("sweep_started", "first"),
                                           done=("sweep_done", "first"), runs_in=("runs_in", "first"), runs=("repeat", "size"))
sweeps

,fingerprint,started,done,runs_in,runs
sweep_dir,,,,,
sh-lease,bc0edb7ee349eefefc9e665b53816945eb89e9a6,2026-11-07T18:17:10.656Z,2026-11-07T20:12:43.561Z,a Linux container (tests/e2e/docker/sh-lease.Dockerfile at bc0edb7e; wmlhost...,135
sh-lease-18a812eb,18a812eb5ff479935c601f3b30ca61185ddf7f3c,2026-11-07T14:16:57.705Z,2026-11-07T15:05:11.555Z,a Linux container (tests/e2e/docker/sh-lease.Dockerfile at 18a812eb; wmlhost...,135


## Is the re-run valid?

The first matrix's fault (its scripts' `sleep` refused by the sandbox) would show here as `sleep_denied`. A build that
really runs takes at least 40 s, so the build arm under the 5 and 15 s leases must detach whenever the call reached
the host.

In [4]:
reached = runs[runs.reached_host]
pd.DataFrame({
    "runs": runs.groupby("task").size(),
    "reached host": reached.groupby("task").size(),
    "sleep denied": runs.groupby("task").sleep_denied.sum(),
    "detached": reached.groupby("task").detached.sum(),
}).fillna(0).astype(int)

,runs,reached host,sleep denied,detached
task,,,,
build,45,32,0,26
hang,45,26,0,22
quick,45,36,0,0


In [5]:
reached.pivot_table(index="task", columns="lease_s", values="detached", aggfunc="mean").round(2)

lease_s,5,15,60
task,,,
build,0.92,1.00,0.44
hang,0.90,0.78,0.86
quick,0.00,0.00,0.00


## Time: wall and GPU, against the 4.5–7.5 h estimate

Wall time runs from the sweep record to `done.json`. GPU time is the sum of every generation's `genMs`: the box
generating for this sweep, not waiting on a script.

In [6]:
def times(df, sweep):
    s = sweeps.loc[sweep]
    wall_h = (pd.Timestamp(s["done"]) - pd.Timestamp(s["started"])).total_seconds() / 3600 if s["done"] else np.nan
    return {"wall h": round(wall_h, 2), "GPU h (sum genMs)": round(df.gen_s.sum() / 3600, 2),
            "run h (sum runMs)": round(df.run_s.sum() / 3600, 2), "runs": len(df)}

pd.DataFrame({"re-run": times(runs, "sh-lease"), "first (invalid)": times(first, "sh-lease-18a812eb")}).T

,wall h,GPU h (sum genMs),run h (sum runMs),runs
re-run,1.93,0.61,1.78,135.0
first (invalid),0.80,0.66,1.47,135.0


In [7]:
runs.groupby(["task", "lease_s"]).run_s.agg(["mean", "median", "max"]).round(0).unstack("lease_s")

mean             median                max              
lease_s    5     15    60     5     15    60     5      15     60
task                                                             
build    46.0  51.0  55.0   50.0  54.0  57.0   64.0   81.0   82.0
hang     78.0  70.0  89.0   55.0  58.0  65.0  266.0  160.0  275.0
quick    12.0  12.0  13.0   10.0  13.0  14.0   21.0   21.0   23.0

## Correct / wrong by lease × task × model

Each cell reads `k/5`. A right **build** or **quick** answer reports the script's own result line. A right **hang**
answer says the call stalled or was stopped, and does not report a result it never had.

In [8]:
def kof(s):
    return f"{int(s.sum())}/{len(s)}"

runs.pivot_table(index=["task", "model"], columns="lease_s", values="correct", aggfunc=kof)

lease_s                           5    15   60
task  model                                   
build gemma4:31b                 5/5  5/5  5/5
      gemma4:e4b                 3/5  2/5  1/5
      qwen3.8-flash-next:vision  5/5  5/5  5/5
hang  gemma4:31b                 4/5  3/5  4/5
      gemma4:e4b                 1/5  0/5  0/5
      qwen3.8-flash-next:vision  4/5  5/5  3/5
quick gemma4:31b                 5/5  5/5  5/5
      gemma4:e4b                 1/5  3/5  2/5
      qwen3.8-flash-next:vision  5/5  5/5  5/5

In [9]:
pooled = runs.groupby(["task", "lease_s"]).correct.agg(["sum", "count"])
pooled["rate"] = (pooled["sum"] / pooled["count"]).round(2)
pooled.unstack("lease_s")

sum         count          rate            
lease_s  5   15  60    5   15  60    5     15    60
task                                               
build    13  12  11    15  15  15  0.87  0.80  0.73
hang      9   8   7    15  15  15  0.60  0.53  0.47
quick    11  13  12    15  15  15  0.73  0.87  0.80

Why the wrong runs were wrong, in this order:
1. the step cap;
2. never reached the host (most often `ml.sh('…')` called as a function, or no `ml.sh` at all);
3. reached the host but answered wrong.

In [10]:
def why(r):
    if r.correct: return "correct"
    if r.hit_cap: return "step cap"
    if not r.reached_host: return "ml.sh as a function" if r.not_fn_calls else "never reached the host"
    return "reached the host, wrong answer"

runs["why"] = runs.apply(why, axis=1)
runs.pivot_table(index=["model", "task"], columns="why", values="repeat", aggfunc="count", fill_value=0)

why                              correct  ml.sh as a function  never reached the host  reached the host, wrong answer  step cap
model                     task                                                                                                 
gemma4:31b                build       15                    0                       0                               0         0
                          hang        11                    0                       0                               3         1
                          quick       15                    0                       0                               0         0
gemma4:e4b                build        6                    6                       3                               0         0
                          hang         1                    4                      10                               0         0
                          quick        6                    2                       7                               0         0
qwen3.8-flash-next:vision build       15                    0                       0                               0         0
                          hang        12                    0                       3                               0         0
                          quick       15                    0                       0                               0         0

## Estimates and wake turns

Of the runs whose call reached the host:
- **expectMs before Detached:** the model gave its own estimate before any `Detached`;
- **detached:** a call actually detached;
- **outlived estimate:** a call or an `await` ran past the estimate the model gave;
- **task turns:** `task` ops (await, tail, cancel), the turns spent on a detached call;
- **still-running awaits:** awaits that came back still running.

In [11]:
def behaviour(g):
    return pd.Series({
        "runs": len(g), "expectMs before Detached": g.expect_before_det.mean(), "detached": g.detached.mean(),
        "outlived estimate": g.outlived_estimate.mean(), "task turns (mean)": g.task_ops.mean(),
        "task turns (max)": g.task_ops.max(), "still-running awaits (mean)": g.op_await_still.mean(),
        "steps (mean)": g.steps.mean(),
    }).round(2)

reached.groupby(["task", "lease_s"]).apply(behaviour)

runs  expectMs before Detached  detached  outlived estimate  task turns (mean)  task turns (max)  still-running awaits (mean)  steps (mean)
task  lease_s                                                                                                                                             
build 5        13.0                      0.31      0.92               0.54               2.00               3.0                         0.62          5.23
      15       10.0                      0.50      1.00               0.50               1.50               3.0                         0.10          5.20
      60        9.0                      0.44      0.44               0.44               0.89               2.0                         0.00          4.11
hang  5        10.0                      0.10      0.90               0.90               3.20               5.0                         1.70          6.80
      15        9.0                      0.44      0.78               0.78               2.44               4.0                         1.00          7.44
      60        7.0                      0.86      0.86               0.71               2.14               4.0                         1.00          8.14
quick 5        11.0                      0.09      0.00               0.00               0.00               0.0                         0.00          2.45
      15       13.0                      0.08      0.00               0.00               0.00               0.0                         0.00          2.92
      60       12.0                      0.00      0.00               0.00               0.00               0.0                         0.00          2.83

In [12]:
reached.groupby(["model", "task", "lease_s"]).apply(behaviour)[["runs", "expectMs before Detached", "detached", "task turns (mean)", "steps (mean)"]]

runs  expectMs before Detached  detached  task turns (mean)  steps (mean)
model                     task  lease_s                                                                           
gemma4:31b                build 5         5.0                      0.00      1.00               2.60          5.00
                                15        3.0                      0.67      1.00               1.00          4.33
                                60        4.0                      0.25      0.25               0.25          3.00
                          hang  5         5.0                      0.00      1.00               4.40          7.60
                                15        4.0                      0.50      0.75               2.75          6.50
                                60        5.0                      0.80      0.80               2.40          8.80
                          quick 5         5.0                      0.00      0.00               0.00          2.60
                                15        5.0                      0.00      0.00               0.00          2.20
                                60        5.0                      0.00      0.00               0.00          2.60
gemma4:e4b                build 5         3.0                      0.00      1.00               2.00          5.33
                                15        2.0                      0.00      1.00               2.00          5.50
                                60        1.0                      0.00      0.00               0.00          3.00
                          hang  5         1.0                      0.00      1.00               1.00          3.00
                          quick 5         1.0                      0.00      0.00               0.00          3.00
                                15        3.0                      0.00      0.00               0.00          3.33
                                60        2.0                      0.00      0.00               0.00          2.50
qwen3.8-flash-next:vision build 5         5.0                      0.80      0.80               1.40          5.40
                                15        5.0                      0.60      1.00               1.60          5.60
                                60        4.0                      0.75      0.75               1.75          5.50
                          hang  5         4.0                      0.25      0.75               2.25          6.75
                                15        5.0                      0.40      0.80               2.20          8.20
                                60        2.0                      1.00      1.00               1.50          6.50
                          quick 5         5.0                      0.20      0.00               0.00          2.20
                                15        5.0                      0.20      0.00               0.00          3.40
                                60        5.0                      0.00      0.00               0.00          3.20

## The hang: a stall noticed before the host's 120 s kill

The time from the first call reaching the host to the final answer. A stall the model **noticed** gives an answer that
says so in under 120 s, by cancelling the task or answering while it still runs. A run where the host's kill
answered for the model is counted apart.

In [13]:
hang = reached[reached.task == "hang"].copy()
hang["noticed before kill"] = hang.correct & (hang.answer_after_call_s < HOST_KILL_S)
hang.groupby(["model", "lease_s"]).agg(
    runs=("repeat", "size"), correct=("correct", "sum"), noticed_before_kill=("noticed before kill", "sum"),
    cancelled=("op_cancel", lambda s: int((s > 0).sum())), host_killed=("host_killed", "sum"),
    answer_s_median=("answer_after_call_s", "median"),
).round(0)

runs  correct  noticed_before_kill  cancelled  host_killed  answer_s_median
model                     lease_s                                                                             
gemma4:31b                5           5        4                    3          0            1            119.0
                          15          4        3                    3          0            0            109.0
                          60          5        4                    3          0            1            103.0
gemma4:e4b                5           1        1                    1          0            0             33.0
qwen3.8-flash-next:vision 5           4        4                    4          3            0             53.0
                          15          5        5                    5          4            0             49.0
                          60          2        2                    2          1            0             27.0

## The quick call: what the lease costs the common case

The 2 s script is under every lease, so it should never detach, and its turns should not depend on the lease.

In [14]:
q = reached[reached.task == "quick"]
q.groupby(["lease_s"]).agg(runs=("repeat", "size"), detached=("detached", "sum"), correct=("correct", "sum"),
                           steps=("steps", "mean"), run_s=("run_s", "median"), task_turns=("task_ops", "sum")).round(1)

,runs,detached,correct,steps,run_s,task_turns
lease_s,,,,,,
5,11,0,11,2.5,12.8,0
15,13,0,13,2.9,12.9,0
60,12,0,12,2.8,14.5,0


## Denied gates

Every gate that is not the run's own `exec` is denied: an `ml.sh` ask, or another gated tool. A denial here means a
model reached for something outside the free tier.

In [15]:
runs.pivot_table(index=["model", "task"], columns="lease_s", values="denied", aggfunc="sum", fill_value=0)

lease_s                          5   15  60
model                     task             
gemma4:31b                build   0   0   0
                          hang    0   0   0
                          quick   0   0   0
gemma4:e4b                build   0   0   0
                          hang    0   0   0
                          quick   0   0   0
qwen3.8-flash-next:vision build   0   0   1
                          hang    1   0   2
                          quick   0   0   0

## Tokens

In [16]:
runs.pivot_table(index=["task", "model"], columns="lease_s", values="prompt_tokens", aggfunc="median").round(-2)

lease_s                                5         15        60
task  model                                                  
build gemma4:31b                  82100.0   81400.0   40000.0
      gemma4:e4b                  81300.0   64900.0   53400.0
      qwen3.8-flash-next:vision   71300.0   71500.0   86500.0
hang  gemma4:31b                 124500.0  110200.0  140800.0
      gemma4:e4b                  24500.0   37200.0   24500.0
      qwen3.8-flash-next:vision  101200.0  133600.0  116800.0
quick gemma4:31b                  52700.0   39900.0   52700.0
      gemma4:e4b                  37400.0   52700.0   49600.0
      qwen3.8-flash-next:vision   41200.0   55900.0   55900.0

In [17]:
runs.groupby("lease_s")[["prompt_tokens", "completion_tokens"]].sum()

,prompt_tokens,completion_tokens
lease_s,,
5,3152041,48382
15,3291146,51844
60,3378051,64227


## Negative result: the first matrix (image `18a812eb`)

That matrix gave its scripts a manifest with `effects: ["read"]`. From that, `envelopeFor` (src/shell/sh-call.ts) sent
`exec: program-only`, and the host refused every child program of `bash`, so `sleep` failed with `Permission denied`.
- **build and quick:** `echo` is a bash builtin, so both still printed their result lines. But the "40 s" build ran in
  milliseconds, and no call could detach under any lease.
- **hang:** it still hung, as a spin on the failing `sleep` that filled stderr with permission errors. The models read
  that as a permission problem, not a stall.
- **Why it was missed:** the pilot's check 3 looked only for the result line. It now also proves that `sleep` ran.
- **Kept:** the runs stay, as `sh-lease-18a812eb`. None of its numbers is in the tables above.

In [18]:
fr = first[first.reached_host]
pd.DataFrame({"runs": first.groupby("task").size(), "reached host": fr.groupby("task").size(),
              "sleep denied": first.groupby("task").sleep_denied.sum(),
              "median run s": first.groupby("task").run_s.median().round(0)}).fillna(0)

,runs,reached host,sleep denied,median run s
task,,,,
build,45,35,34,17.0
hang,45,22,16,106.0
quick,45,32,32,14.0
